## Replicando o que foi feito no inicio do primeiro arquivo dessa análise

In [35]:
# Importando o pandas
import pandas as pd

In [36]:
# Visualizando a base de treino
treino = pd.read_csv('train.csv')
treino.head(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [37]:
# Visualizando a base de teste
teste = pd.read_csv('test.csv')
treino.head(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [38]:
# Eliminando colunas com elevada cardinalidade
treino = treino.drop(['Name', 'Ticket', 'Cabin'], axis=1)
teste = teste.drop(['Name', 'Ticket', 'Cabin'], axis=1)

In [39]:
# Usando a média para substituir valores nulos na coluna de idade
treino.loc[treino.Age.isnull(), 'Age'] = treino.Age.mean()
teste.loc[teste.Age.isnull(), 'Age'] = teste.Age.mean()

In [40]:
# Tratando a coluna Embarked da base de treino usando a moda
treino.loc[treino.Embarked.isnull(), 'Embarked'] = treino.Embarked.mode()[0]

In [41]:
# E também a coluna Fare da base de teste usando a média
teste.loc[teste.Fare.isnull(), 'Fare'] = teste.Fare.mean()

## Tratamento das colunas de texto

In [42]:
# Verificando as colunas de texto na base de treino
treino.columns[treino.dtypes == 'object']

Index(['Sex', 'Embarked'], dtype='object')

In [43]:
# Verificando valores na coluna Sex
treino.Sex.value_counts()

Sex
male      577
female    314
Name: count, dtype: int64

In [44]:
# Verificando valores na coluna Embarked
treino.Embarked.value_counts()

Embarked
S    646
C    168
Q     77
Name: count, dtype: int64

**Observação:** Para tratar a coluna sex, foi criado uma nova coluna chamada "MaleCheck" que vai receber 1 se o gênero for masculino e 0 se o gênero for feminino

In [45]:
# Usando uma lambda function para fazer esse tratamento
treino['MaleCheck'] = treino.Sex.apply(lambda x: 1 if x == 'male' else  0)

In [46]:
# Verificando os valores
treino[['Sex', 'MaleCheck']].value_counts()

Sex     MaleCheck
male    1            577
female  0            314
Name: count, dtype: int64

In [47]:
# Lambda function para a base de teste
teste['MaleCheck'] = teste.Sex.apply(lambda x: 1 if x == 'male' else  0)

In [48]:
# Verificando os valores
teste[['Sex', 'MaleCheck']].value_counts()

Sex     MaleCheck
male    1            266
female  0            152
Name: count, dtype: int64

**Obeservação:** Agora, para tratar a coluna Embarked usando o OneHotEncoder que irá criar uma nova coluna para cada um dos rótulos da coluna original

In [49]:
# Importando o OneHotEncoder
from sklearn.preprocessing import OneHotEncoder

In [34]:
# Criando o encoder
ohe = OneHotEncoder(handle_unknown='ignore', dtype='int32')

In [50]:
# Fazendo o fit com os dados
ohe = ohe.fit(treino[['Embarked']])

In [51]:
# Fazendo a transformação
ohe.transform(treino[['Embarked']]).toarray()

array([[0, 0, 1],
       [1, 0, 0],
       [0, 0, 1],
       ...,
       [0, 0, 1],
       [1, 0, 0],
       [0, 1, 0]], shape=(891, 3), dtype=int32)

In [52]:
# Transformando esse resultado em um DataFrame
ohe_df = pd.DataFrame(ohe.transform(treino[['Embarked']]).toarray(), columns=ohe.get_feature_names_out())
ohe_df.head(3)

,Embarked_C,Embarked_Q,Embarked_S
0,0,0,1
1,1,0,0
2,0,0,1


In [53]:
# Adicionando essa coluna na base de treino
treino = pd.concat([treino, ohe_df], axis=1)

In [54]:
# Verificando valores
treino[['Embarked', 'Embarked_C', 'Embarked_Q',	'Embarked_S']].value_counts()

Embarked  Embarked_C  Embarked_Q  Embarked_S
S         0           0           1             646
C         1           0           0             168
Q         0           1           0              77
Name: count, dtype: int64

### Fazendo o mesmo para a base de teste, usando o encoder criado acima

In [55]:
# Transformando esse resultado em um DataFrame
ohe_df = pd.DataFrame(ohe.transform(teste[['Embarked']]).toarray(), columns=ohe.get_feature_names_out())

In [56]:
# Adicionando o resultado na base de teste
teste = pd.concat([teste, ohe_df], axis=1)

In [57]:
# Verificando os valores
teste[['Embarked', 'Embarked_C', 'Embarked_Q',	'Embarked_S']].value_counts()

Embarked  Embarked_C  Embarked_Q  Embarked_S
S         0           0           1             270
C         1           0           0             102
Q         0           1           0              46
Name: count, dtype: int64

## Usando essa nova base no modelo

In [58]:
# Visualizando a base
treino.head(3)

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,MaleCheck,Embarked_C,Embarked_Q,Embarked_S
0,1,0,3,male,22.0,1,0,7.2500,S,1,0,0,1
1,2,1,1,female,38.0,1,0,71.2833,C,0,1,0,0
2,3,1,3,female,26.0,0,0,7.9250,S,0,0,0,1


In [59]:
# Removendo colunas ja tratadas
treino = treino.drop(['Sex', 'Embarked'], axis=1)
teste = teste.drop(['Sex', 'Embarked'], axis=1)

In [60]:
# Importando o train_test_split
from sklearn.model_selection import train_test_split

In [61]:
# Separando a base de treino em x e y
X = treino.drop(['PassengerId', 'Survived'], axis=1)
y = treino.Survived 

In [62]:
# Separando em treino e validação
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.33, random_state=42)

### Arvore de classificação

In [63]:
# Fazendo a importação
from sklearn import tree

In [64]:
# Criando o classificador
clf_ac = tree.DecisionTreeClassifier(random_state=42)

In [65]:
# Fazendo fit com os dados
clf_ac = clf_ac.fit(X_train, y_train)

In [66]:
# Fazendo a previsão
y_pred_ac = clf_ac.predict(X_val)

### KNeighborsClassifier

In [67]:
# Importando
from sklearn.neighbors import KNeighborsClassifier

In [68]:
# Criando o classificador
clf_knn = KNeighborsClassifier(n_neighbors=3)

In [69]:
# Fazendo o fit com os dados
clf_knn = clf_knn.fit(X_train, y_train)

In [71]:
# Fazendo a previsão
y_pred_knn = clf_knn.predict(X_val)

### Regressão Logística

In [72]:
# Importando
from sklearn.linear_model import LogisticRegression

In [75]:
# Criando o classificador
clf_r1 = LogisticRegression(random_state=42, max_iter=1000)

In [76]:
# Fazendo o fit com os dados
clf_r1 = clf_r1.fit(X_train, y_train)

In [77]:
# Fazendo a previsão
y_pred_r1 = clf_r1.predict(X_val)

## Avaliando os modelos

### Acurácia

In [78]:
# Importando
from sklearn.metrics import accuracy_score

In [79]:
# Para a árvore
accuracy_score(y_val, y_pred_ac)

0.7491525423728813

In [80]:
# Para o Knn
accuracy_score(y_val, y_pred_knn)

0.7152542372881356

In [81]:
# Para a Regressão Logística
accuracy_score(y_val, y_pred_r1)

0.8169491525423729

### Matriz de confusão

In [82]:
# Importando
from sklearn.metrics import confusion_matrix

In [83]:
# Para a árvore
confusion_matrix(y_val, y_pred_ac)

array([[138,  37],
       [ 37,  83]])

In [84]:
# Para o Knn
confusion_matrix(y_val, y_pred_knn)

array([[147,  28],
       [ 56,  64]])

In [85]:
# Para a Regressão Logística
confusion_matrix(y_val, y_pred_r1)

array([[153,  22],
       [ 32,  88]])

## Previsão para os dados de teste

In [86]:
# Visualizando o X_train
X_train.head(3)

,Pclass,Age,SibSp,Parch,Fare,MaleCheck,Embarked_C,Embarked_Q,Embarked_S
6,1,54.000000,0,0,51.8625,1,0,0,1
718,3,29.699118,0,0,15.5000,1,0,1,0
685,2,25.000000,1,2,41.5792,1,1,0,0


In [87]:
# Visualizando a base de teste
teste.head(3)

,PassengerId,Pclass,Age,SibSp,Parch,Fare,MaleCheck,Embarked_C,Embarked_Q,Embarked_S
0,892,3,34.5,0,0,7.8292,1,0,1,0
1,893,3,47.0,1,0,7.0000,0,0,0,1
2,894,2,62.0,0,0,9.6875,1,0,1,0


In [90]:
# Para a base de teste ser igual a base de treino, precisamos eliminar a coluna de id
X_teste = teste.drop('PassengerId', axis=1)

In [91]:
# Utilizando a regressão logistica na base de teste
y_pred = clf_r1.predict(X_teste)

In [92]:
# Criando uma nova coluna com a previsão na abse de teste
teste['Survived'] = y_pred

In [95]:
# Selecionando apenas a coluna de Id e Survived para fazer o envio
base_envio = teste[['PassengerId', 'Survived']]

In [97]:
# Exportando para um csv
base_envio.to_csv('resultados2.csv', index=False)